# 09 — Monthly Dataset Integration

**Project:** Dengue Disease Prediction — Sri Lanka  
**Purpose:** Join validated monthly dengue, weather and population data using the same administrative-district and month keys.  
**Scope:** Data integration and integrity checks only. Exploratory analysis, temporal features, train/test splits and LSTM fitting belong in later notebooks.

> **Important:** The 2024 forecasting test period is not used to choose preprocessing methods. Joining data here does not train or tune a model. The original weather data has no natural internal missing values, so the artificial masks created in Notebook 08 must **not** be carried into this integrated dataset.

**Expected outputs:**
- Full dengue + weather: 2010–2024, **4,500 district-months**.
- Population-inclusive: 2014–2024, **3,300 district-months**.
- Population-inclusive with complete weather-month coverage: **3,275 district-months**, excluding December 2024.

Expected counts are based on the earlier notebooks. If your source data has changed, investigate differences rather than forcing these values.

## Step 1 — Imports and locate project root

The notebook should run from the project `notebooks/` folder in VS Code. We search the current directory and its parents so it also works when VS Code starts in the project root.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT_CANDIDATES = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    (p for p in ROOT_CANDIDATES if (p / "data" / "interim").is_dir()),
    None
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Cannot find the project root. Open D:\\Dengue_Project in VS Code "
        "and run this notebook from its notebooks/ folder."
    )

INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TABLES_DIR = PROJECT_ROOT / "results" / "tables"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)
print("Project root:", PROJECT_ROOT)

Project root: d:\Dengue_Project


## Step 2 — Load the three prepared datasets

Only use the prepared **interim** CSV files from Notebooks 04–06. Do not reload the raw weekly CSVs or the artificially masked experiments from Notebook 08.

In [2]:
source_paths = {
    "dengue": INTERIM_DIR / "dengue_monthly_proportional_2010_2024.csv",
    "weather": INTERIM_DIR / "weather_monthly_prepared_2010_2024.csv",
    "population": INTERIM_DIR / "population_monthly_prepared_2014_2024.csv",
}
for label, path in source_paths.items():
    print(f"{label:11} exists={path.exists()}  {path}")
    if not path.is_file():
        raise FileNotFoundError(f"Missing {label} input: {path}")

dengue = pd.read_csv(source_paths["dengue"], parse_dates=["date"])
weather = pd.read_csv(source_paths["weather"], parse_dates=["date"])
population = pd.read_csv(source_paths["population"], parse_dates=["date"])
for name, df in [("Dengue", dengue), ("Weather", weather), ("Population", population)]:
    print(f"{name:11} shape={df.shape}, months={df['date'].nunique()}, districts={df['administrative_district'].nunique()}")

dengue      exists=True  d:\Dengue_Project\data\interim\dengue_monthly_proportional_2010_2024.csv
weather     exists=True  d:\Dengue_Project\data\interim\weather_monthly_prepared_2010_2024.csv
population  exists=True  d:\Dengue_Project\data\interim\population_monthly_prepared_2014_2024.csv
Dengue      shape=(4500, 5), months=180, districts=25
Weather     shape=(4500, 16), months=180, districts=25
Population  shape=(3300, 5), months=132, districts=25


## Step 3 — Check required columns, dates, and merge keys

The merge key is **(`administrative_district`, `date`)**, not the original 26-area reporting geography. `Ampara` and `Kalmunai` have already been handled in the earlier standardization and aggregation notebooks.

Fail loudly if any source has duplicate keys or inconsistent calendar fields.

In [3]:
KEYS = ["administrative_district", "date"]
required = {
    "dengue": {"administrative_district", "date", "dengue_cases"},
    "weather": {"administrative_district", "date", "complete_weather_month",
                "temperature_mean_c", "humidity_mean_percent", "rainfall_total_mm", "wind_speed_mean_kmh"},
    "population": {"administrative_district", "date", "population"},
}
for name, df in [("dengue", dengue), ("weather", weather), ("population", population)]:
    absent = required[name] - set(df.columns)
    if absent:
        raise ValueError(f"{name} missing required columns: {sorted(absent)}")
    if df[KEYS].isna().any().any():
        raise ValueError(f"{name} has null merge keys")
    if df.duplicated(KEYS).any():
        raise ValueError(f"{name} has duplicate district-month keys")
    if not df["date"].dt.is_month_start.all():
        raise ValueError(f"{name} has non-month-start dates")
    if "year" in df.columns and not df["year"].eq(df["date"].dt.year).all():
        raise ValueError(f"{name} has inconsistent year values")
    if "month_number" in df.columns and not df["month_number"].eq(df["date"].dt.month).all():
        raise ValueError(f"{name} has inconsistent month_number values")
    print(f"{name:11}: unique district-month keys verified ({len(df):,})")

assert len(dengue) == 4500, "Unexpected dengue rows; review Notebook 04"
assert len(weather) == 4500, "Unexpected weather rows; review Notebook 05"
assert len(population) == 3300, "Unexpected population rows; review Notebook 06"

dengue     : unique district-month keys verified (4,500)
weather    : unique district-month keys verified (4,500)
population : unique district-month keys verified (3,300)


## Step 4 — Normalize the weather completeness flag

`complete_weather_month` identifies partial calendar-month coverage, **not** an ordinary NaN. Earlier notebooks identified January 2010 and December 2024 as partial boundary months.

In [4]:
weather = weather.copy()
flag = weather["complete_weather_month"].astype("string").str.strip().str.lower()
flag_map = {"true": True, "false": False, "1": True, "0": False}
weather["complete_weather_month"] = flag.map(flag_map)
if weather["complete_weather_month"].isna().any():
    raise ValueError("Unrecognized complete_weather_month values")
weather["complete_weather_month"] = weather["complete_weather_month"].astype(bool)

print(weather.groupby(["date", "complete_weather_month"]).size().loc[
    lambda x: x.index.get_level_values("complete_weather_month") == False
])

date        complete_weather_month
2010-01-01  False                     25
2024-12-01  False                     25
dtype: int64


## Step 5 — Join dengue with weather (full 2010–2024 coverage)

Both datasets have one row per district and month. We use an explicit one-to-one merge and check that no observations were lost.

In [5]:
weather_fields = [col for col in weather.columns if col not in {"year", "month_number"}]
dengue_weather = dengue[["date", "administrative_district", "dengue_cases"]].merge(
    weather[weather_fields],
    on=KEYS,
    how="inner",
    validate="one_to_one",
    indicator=True
).sort_values(KEYS).reset_index(drop=True)

assert (dengue_weather["_merge"] == "both").all()
dengue_weather = dengue_weather.drop(columns="_merge")
assert len(dengue_weather) == len(dengue) == len(weather) == 4500
assert dengue_weather["dengue_cases"].notna().all()
assert dengue_weather["administrative_district"].nunique() == 25
assert dengue_weather["date"].nunique() == 180

print("Dengue + weather:", dengue_weather.shape)
print("Date range:", dengue_weather["date"].min(), "to", dengue_weather["date"].max())
display(dengue_weather.head())

Dengue + weather: (4500, 15)
Date range: 2010-01-01 00:00:00 to 2024-12-01 00:00:00


,date,administrative_district,dengue_cases,temperature_mean_c,temperature_min_c,temperature_max_c,humidity_mean_percent,humidity_min_percent,humidity_max_percent,rainfall_total_mm,rainfall_max_hourly_mm,rainy_days,wind_speed_mean_kmh,wind_speed_max_kmh,complete_weather_month
0,2010-01-01,Ampara,116.428571,25.599833,22.20,28.95,80.270667,59.5,96.5,88.371429,4.35,21.714286,12.701000,24.15,False
1,2010-02-01,Ampara,124.428571,26.125893,22.20,31.40,78.548750,56.0,94.0,72.071429,4.35,12.571429,11.530714,24.15,True
2,2010-03-01,Ampara,72.857143,27.752097,22.55,32.55,75.846774,51.0,94.0,25.885714,2.45,14.500000,9.655000,21.65,True
3,2010-04-01,Ampara,37.285714,28.550833,25.20,32.80,79.705500,57.5,95.5,73.921429,3.45,28.214286,7.864833,16.35,True
4,2010-05-01,Ampara,13.714286,28.975968,25.45,34.20,79.241935,57.0,94.5,79.442857,4.95,28.000000,8.257419,28.35,True


## Step 6 — Join population without inventing 2010–2013 values

A left join retains all 4,500 dengue-weather observations. Missing population values before 2014 are a **structural coverage gap**, not isolated missing observations to fill with KNN or interpolation.

In [6]:
integrated_full = dengue_weather.merge(
    population[KEYS + ["population"]],
    on=KEYS,
    how="left",
    validate="one_to_one",
    indicator="population_merge_status"
).sort_values(KEYS).reset_index(drop=True)

integrated_full["year"] = integrated_full["date"].dt.year
integrated_full["month_number"] = integrated_full["date"].dt.month
integrated_full["population_available"] = integrated_full["population"].notna()

print("Full joined dataset:", integrated_full.shape)
print("Population available:", int(integrated_full["population_available"].sum()))
print("Population unavailable:", int((~integrated_full["population_available"]).sum()))
print("Population-unavailable years:", sorted(integrated_full.loc[
    ~integrated_full["population_available"], "year"].unique().tolist()))

assert len(integrated_full) == 4500
assert integrated_full["population_available"].sum() == 3300
assert (~integrated_full["population_available"]).sum() == 1200
assert set(integrated_full.loc[~integrated_full["population_available"], "year"]) == {2010, 2011, 2012, 2013}
assert integrated_full.loc[integrated_full["year"] >= 2014, "population"].notna().all()

Full joined dataset: (4500, 20)
Population available: 3300
Population unavailable: 1200
Population-unavailable years: [2010, 2011, 2012, 2013]


## Step 7 — Produce analysis-ready dataset variants

**Variant A:** Dengue + weather, 2010–2024, useful when population is not used as an input.  
**Variant B:** Dengue + weather + population, 2014–2024, preserving the `complete_weather_month` indicator.  
**Variant C:** Population-inclusive and complete weather months only, 2014–November 2024. This is a conservative candidate for modeling; later notebooks must still define the forecasting target and time split.

Do not automatically delete December 2024 from all datasets. Retain it with the coverage flag for traceability.

In [7]:
dengue_weather_full = dengue_weather.copy()
population_inclusive = (
    integrated_full.loc[integrated_full["population_available"]]
    .drop(columns=["population_merge_status"])
    .copy()
    .reset_index(drop=True)
)
population_strict = (
    population_inclusive.loc[population_inclusive["complete_weather_month"]]
    .copy()
    .reset_index(drop=True)
)

print("A. Dengue + weather:", len(dengue_weather_full))
print("B. Population-inclusive:", len(population_inclusive))
print("C. Population + complete weather:", len(population_strict))
print("Strict last date:", population_strict["date"].max())

assert len(dengue_weather_full) == 4500
assert len(population_inclusive) == 3300
assert len(population_strict) == 3275
assert population_strict["date"].max() == pd.Timestamp("2024-11-01")

A. Dengue + weather: 4500
B. Population-inclusive: 3300
C. Population + complete weather: 3275
Strict last date: 2024-11-01 00:00:00


## Step 8 — Final data integrity checks

Check row uniqueness, nonnegative cases, population coverage, and weather variable completeness. No modeling, imputation, or future-looking temporal features are performed here.

In [8]:
weather_variables = ["temperature_mean_c", "humidity_mean_percent", "rainfall_total_mm", "wind_speed_mean_kmh"]
for label, frame in [
    ("Dengue-weather", dengue_weather_full),
    ("Population-inclusive", population_inclusive),
    ("Population-strict", population_strict)
]:
    assert not frame.duplicated(KEYS).any(), f"Duplicate district-month keys in {label}"
    assert frame["dengue_cases"].notna().all()
    assert frame["dengue_cases"].ge(0).all()
    assert frame[weather_variables].notna().all().all()
    assert frame["administrative_district"].nunique() == 25
    if "population" in frame.columns:
        assert frame["population"].notna().all()
        assert frame["population"].gt(0).all()
    print(f"PASS: {label} ({len(frame):,} district-months)")

# Verify the dengue case totals did not change during joining.
assert np.isclose(dengue_weather_full["dengue_cases"].sum(), dengue["dengue_cases"].sum())
print("PASS: Total dengue cases preserved")

PASS: Dengue-weather (4,500 district-months)
PASS: Population-inclusive (3,300 district-months)
PASS: Population-strict (3,275 district-months)
PASS: Total dengue cases preserved


## Step 9 — Summarize annual availability

This summary makes the pre-2014 population gap and boundary-month weather coverage visible to readers of the research.

In [9]:
annual_coverage = (
    integrated_full.groupby("year", as_index=False)
    .agg(
        district_months=("date", "size"),
        population_available=("population_available", "sum"),
        complete_weather_months=("complete_weather_month", "sum")
    )
)
annual_coverage["population_unavailable"] = (
    annual_coverage["district_months"] - annual_coverage["population_available"]
)
annual_coverage["incomplete_weather_months"] = (
    annual_coverage["district_months"] - annual_coverage["complete_weather_months"]
)
display(annual_coverage)

summary = pd.DataFrame([
    {"dataset": "dengue_weather_full", "rows": len(dengue_weather_full), "districts": 25,
     "first_month": dengue_weather_full["date"].min().date().isoformat(),
     "last_month": dengue_weather_full["date"].max().date().isoformat(),
     "population_required": False, "complete_weather_required": False},
    {"dataset": "population_inclusive", "rows": len(population_inclusive), "districts": 25,
     "first_month": population_inclusive["date"].min().date().isoformat(),
     "last_month": population_inclusive["date"].max().date().isoformat(),
     "population_required": True, "complete_weather_required": False},
    {"dataset": "population_strict", "rows": len(population_strict), "districts": 25,
     "first_month": population_strict["date"].min().date().isoformat(),
     "last_month": population_strict["date"].max().date().isoformat(),
     "population_required": True, "complete_weather_required": True}
])
display(summary)

,year,district_months,population_available,complete_weather_months,population_unavailable,incomplete_weather_months
0,2010,300,0,275,300,25
1,2011,300,0,300,300,0
2,2012,300,0,300,300,0
3,2013,300,0,300,300,0
4,2014,300,300,300,0,0
5,2015,300,300,300,0,0
6,2016,300,300,300,0,0
7,2017,300,300,300,0,0
8,2018,300,300,300,0,0
9,2019,300,300,300,0,0


,dataset,rows,districts,first_month,last_month,population_required,complete_weather_required
0,dengue_weather_full,4500,25,2010-01-01,2024-12-01,False,False
1,population_inclusive,3300,25,2014-01-01,2024-12-01,True,False
2,population_strict,3275,25,2014-01-01,2024-11-01,True,True


## Step 10 — Save the three dataset variants and audit tables

Outputs are written to `data/processed/` and `results/tables/`. Original interim files are not modified. Your `.gitignore` may exclude processed datasets; if so, commit the notebook and non-sensitive summary tables rather than overriding that rule.

In [10]:
outputs = {
    PROCESSED_DIR / "dengue_weather_monthly_2010_2024.csv": dengue_weather_full,
    PROCESSED_DIR / "dengue_weather_population_monthly_2014_2024.csv": population_inclusive,
    PROCESSED_DIR / "dengue_weather_population_complete_months_2014_2024.csv": population_strict,
    TABLES_DIR / "notebook09_integration_summary.csv": summary,
    TABLES_DIR / "notebook09_annual_coverage.csv": annual_coverage,
}
for path, frame in outputs.items():
    frame.to_csv(path, index=False)
    print(f"Saved: {path.relative_to(PROJECT_ROOT)} ({len(frame):,} rows)")

assert all(path.is_file() for path in outputs)
print("All Notebook 09 output files saved successfully.")

Saved: data\processed\dengue_weather_monthly_2010_2024.csv (4,500 rows)
Saved: data\processed\dengue_weather_population_monthly_2014_2024.csv (3,300 rows)
Saved: data\processed\dengue_weather_population_complete_months_2014_2024.csv (3,275 rows)
Saved: results\tables\notebook09_integration_summary.csv (3 rows)
Saved: results\tables\notebook09_annual_coverage.csv (15 rows)
All Notebook 09 output files saved successfully.


## Key findings and research decisions

1. Dengue and weather align on **25 administrative districts × 180 months = 4,500 district-months** for 2010–2024.
2. Population is available for **2014–2024 only**, yielding **3,300 population-inclusive district-months**. The 1,200 earlier rows remain unavailable; no synthetic population values are created.
3. Weather boundary-month completeness is recorded explicitly. The conservative population-inclusive complete-weather dataset contains **3,275 rows** and ends in November 2024.
4. Notebook 08's artificial missingness is **not** propagated into the integrated data. The imputation benchmark measures robustness under simulated missingness, not improvements on naturally complete observations.
5. This notebook performs **no temporal feature construction or model selection**. In later notebooks, protect the 2024 forecasting test period and avoid using future dengue values as predictors.
6. The research must decide whether to use the longer **2010–2024 dengue-weather dataset without population**, or the **2014–2024 population-inclusive dataset**, and justify the choice before training the LSTM.

**Next:** Notebook 10 — Exploratory Data Analysis. After that: temporal features, time-aware split, and fixed LSTM.